# 🌡️ Urban Heat Island Planner: Reinforcement Learning Training

Welcome to the **Urban Heat Island Planner** training notebook! This notebook demonstrates how we train an intelligent city planner agent using Reinforcement Learning (specifically, Hugging Face's TRL framework) to mitigate extreme urban heat.

### 🌟 The Hackathon Challenge
Our environment tests the agent on three core themes:
1. **Long-Horizon Planning:** The agent must plant trees early (they take 1 year to grow) to survive the Summer Heatwaves.
2. **World Modeling:** The agent cannot just place interventions magically. It must navigate enterprise-like APIs (`query_zoning`, `propose_budget`, `deploy_intervention`).
3. **Multi-Agent Interactions:** The agent's budget proposals are evaluated by a simulated "Mayor" who has hidden biases (e.g., rejecting projects in low-density zones).

### 🛠️ Notebook Architecture
Because our environment is designed as a robust `FastAPI` server, this notebook will:
1. Clone our environment code from GitHub.
2. Start the FastAPI server silently in the background.
3. Execute our PPO training loop (`train_trl.py`) against the running server.

---
## Step 1: Environment Setup
First, we clone the official repository and install the required dependencies to run both the FastAPI server and the Hugging Face TRL training loop.

In [ ]:
!git clone https://github.com/Shoaibahmed-2005/Urban-Heat-Round-2.git urban_heat_env
%cd urban_heat_env

# Install core environment requirements and TRL libraries
!pip install -r requirements.txt
!pip install trl==0.8.6 peft transformers torch
!cp .env.example .env

In [ ]:
import os
from getpass import getpass

print('Please enter your Hugging Face Token (Get it from https://huggingface.co/settings/tokens):')
os.environ['HF_TOKEN'] = getpass()
os.environ['API_KEY'] = os.environ['HF_TOKEN']


---
## Step 2: Booting the City Server
Our environment is completely decoupled from the training logic. The environment logic lives in `server/environment.py` and is served via `server/app.py`.

We use Python's `subprocess` to spin up the FastAPI backend on `localhost:8000`. The RL agent will send HTTP POST requests to this server to take actions (just like a real microservice architecture!).

In [ ]:
import subprocess
import time

print("Starting FastAPI environment server...")
server_process = subprocess.Popen(['uvicorn', 'server.app:app', '--host', '0.0.0.0', '--port', '8000'])

# Give the server 5 seconds to fully boot before we start bombarding it with API requests
time.sleep(5)  
print("Server is running on localhost:8000!")

---
## Step 3: Run PPO Training
With the environment running, we can now launch our training script. 

The `train_trl.py` script utilizes Hugging Face's `PPOTrainer` to train a model to act as the City Planner. Over multiple episodes, the agent learns to secure budget from the Mayor and strategically plant trees and reflective surfaces to cool the city down.

In [ ]:
# Execute the training script
!python train_trl.py

---
## Step 4: Graceful Shutdown
After training is complete, we terminate the background FastAPI process to free up resources.

In [ ]:
server_process.terminate()
print("Server shut down successfully.")

---
## Step 5: Visualize Training Performance
Let's see how our agent's reward evolved over the training epochs.

In [ ]:
import json
import matplotlib.pyplot as plt
import os

if os.path.exists('train_metrics.json'):
    with open('train_metrics.json', 'r') as f:
        metrics = json.load(f)
    rewards = metrics.get('epoch_rewards', [])

    plt.figure(figsize=(10, 6))
    plt.plot(range(1, len(rewards) + 1), rewards, color='blue')
    plt.title('RL Agent Training Curve (PPO)')
    plt.xlabel('Epochs')
    plt.ylabel('Reward')
    plt.grid(True)
    plt.tight_layout()
    plt.savefig('train_plot.png')
    plt.show()
else:
    print('No training metrics found. Did the training finish?')